In [3]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from rdkit import Chem
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

df = pd.read_csv(r"C:\Users\rjmax\Desktop\mol-property-pred\tox21.csv")

targets = [c for c in df.columns if c not in ["mol_id", "smiles"]]
print("Target coverage:\n")
for t in targets:
    total = df[t].notna().sum()
    toxic = df[t].sum()
    print(f"{t:<20} tested: {total:>4} | toxic: {int(toxic):>3} | toxic%: {toxic/total*100:.1f}%")
    
print(f"Total molecules: {len(df)}")

Target coverage:

NR-AR                tested: 7265 | toxic: 309 | toxic%: 4.3%
NR-AR-LBD            tested: 6758 | toxic: 237 | toxic%: 3.5%
NR-AhR               tested: 6549 | toxic: 768 | toxic%: 11.7%
NR-Aromatase         tested: 5821 | toxic: 300 | toxic%: 5.2%
NR-ER                tested: 6193 | toxic: 793 | toxic%: 12.8%
NR-ER-LBD            tested: 6955 | toxic: 350 | toxic%: 5.0%
NR-PPAR-gamma        tested: 6450 | toxic: 186 | toxic%: 2.9%
SR-ARE               tested: 5832 | toxic: 942 | toxic%: 16.2%
SR-ATAD5             tested: 7072 | toxic: 264 | toxic%: 3.7%
SR-HSE               tested: 6467 | toxic: 372 | toxic%: 5.8%
SR-MMP               tested: 5810 | toxic: 918 | toxic%: 15.8%
SR-p53               tested: 6774 | toxic: 423 | toxic%: 6.2%
Total molecules: 7831


In [4]:
def atom_features(atom):
    return [
        atom.GetAtomicNum(),
        atom.GetDegree(),
        atom.GetFormalCharge(),
        int(atom.GetIsAromatic()),
        int(atom.IsInRing()),
        atom.GetTotalNumHs(),
    ]

def smiles_to_graph_multitask(smiles, labels):
    """
    labels: list of 12 values, each is 0.0, 1.0, or NaN
    Returns a PyG Data object with:
      - x: atom features
      - edge_index: bond connectivity  
      - y: 12 toxicity labels (NaN replaced with -1 as mask signal)
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    node_feats = [atom_features(atom) for atom in mol.GetAtoms()]
    x = torch.tensor(node_feats, dtype=torch.float)
    
    edges = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edges += [[i, j], [j, i]]
    
    if len(edges) == 0:
        return None
    
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    
    # Replace NaN with -1 so we can mask them during training
    clean_labels = [-1.0 if np.isnan(l) else float(l) for l in labels]
    y = torch.tensor(clean_labels, dtype=torch.float)
    
    return Data(x=x, edge_index=edge_index, y=y)

# Build dataset
print("Converting molecules to graphs...")
dataset = []
for _, row in df.iterrows():
    smiles = row["smiles"]
    labels = [row[t] for t in TARGETS]
    graph = smiles_to_graph_multitask(smiles, labels)
    if graph is not None:
        dataset.append(graph)

print(f"Successfully converted: {len(dataset)} molecules")

Converting molecules to graphs...


[14:46:59] WARNING: not removing hydrogen atom without neighbors
[14:47:00] Explicit valence for atom # 8 Al, 6, is greater than permitted
[14:47:00] Explicit valence for atom # 3 Al, 6, is greater than permitted
[14:47:00] Explicit valence for atom # 4 Al, 6, is greater than permitted
[14:47:00] Explicit valence for atom # 4 Al, 6, is greater than permitted
[14:47:01] Explicit valence for atom # 9 Al, 6, is greater than permitted
[14:47:01] Explicit valence for atom # 5 Al, 6, is greater than permitted
[14:47:01] Explicit valence for atom # 16 Al, 6, is greater than permitted
[14:47:02] Explicit valence for atom # 20 Al, 6, is greater than permitted


Successfully converted: 7804 molecules


In [5]:
# Split
train_data, test_data = train_test_split(
    dataset, test_size=0.2, random_state=42
)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

print(f"Train: {len(train_data)} | Test: {len(test_data)}")

# Calculate per-target pos_weights for class imbalance
pos_weights = []
for i, t in enumerate(TARGETS):
    labels = [g.y[i].item() for g in train_data if g.y[i].item() != -1]
    neg = sum(1 for l in labels if l == 0)
    pos = sum(1 for l in labels if l == 1)
    weight = neg / pos if pos > 0 else 1.0
    pos_weights.append(weight)
    print(f"{t:<20} pos_weight: {weight:.1f}")

pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float)

Train: 6243 | Test: 1561
NR-AR                pos_weight: 21.9
NR-AR-LBD            pos_weight: 26.4
NR-AhR               pos_weight: 7.8
NR-Aromatase         pos_weight: 18.8
NR-ER                pos_weight: 6.7
NR-ER-LBD            pos_weight: 18.1
NR-PPAR-gamma        pos_weight: 33.1
SR-ARE               pos_weight: 5.1
SR-ATAD5             pos_weight: 25.9
SR-HSE               pos_weight: 16.2
SR-MMP               pos_weight: 5.5
SR-p53               pos_weight: 14.6


In [6]:
from torch_geometric.nn import GCNConv, global_mean_pool

class MultiTaskGNN(nn.Module):
    def __init__(self, input_dim=6, hidden_dim=128, num_tasks=12):
        super(MultiTaskGNN, self).__init__()
        
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        
        # Shared representation layer
        self.shared = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
        )
        
        # One output head per toxicity target
        self.task_heads = nn.ModuleList([
            nn.Linear(64, 1) for _ in range(num_tasks)
        ])
    
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        
        x = torch.relu(self.conv1(x, edge_index))
        x = torch.dropout(x, p=0.2, train=self.training)
        x = torch.relu(self.conv2(x, edge_index))
        x = torch.dropout(x, p=0.2, train=self.training)
        x = torch.relu(self.conv3(x, edge_index))
        
        x = global_mean_pool(x, batch)
        x = self.shared(x)
        
        # Each head predicts one target
        out = torch.cat([head(x) for head in self.task_heads], dim=1)
        return out  # shape: [batch_size, 12]

model = MultiTaskGNN(input_dim=6, hidden_dim=128, num_tasks=len(TARGETS))
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

MultiTaskGNN(
  (conv1): GCNConv(6, 128)
  (conv2): GCNConv(128, 128)
  (conv3): GCNConv(128, 128)
  (shared): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
  )
  (task_heads): ModuleList(
    (0-11): 12 x Linear(in_features=64, out_features=1, bias=True)
  )
)

Total parameters: 42,956


In [9]:
def masked_loss(predictions, targets, pos_weights):
    # Reshape targets from [batch*12] to [batch, 12] if needed
    if targets.dim() == 1:
        targets = targets.view(-1, len(TARGETS))
    
    total_loss = 0
    count = 0
    
    for i in range(targets.shape[1]):
        target_col = targets[:, i]
        pred_col = predictions[:, i]
        
        mask = target_col != -1
        if mask.sum() == 0:
            continue
        
        masked_pred = pred_col[mask]
        masked_target = target_col[mask]
        
        criterion = nn.BCEWithLogitsLoss(
            pos_weight=pos_weights[i].unsqueeze(0)
        )
        total_loss += criterion(masked_pred, masked_target)
        count += 1
    
    return total_loss / count if count > 0 else torch.tensor(0.0)


def evaluate_multitask(loader):
    model.eval()
    all_preds = [[] for _ in range(len(TARGETS))]
    all_labels = [[] for _ in range(len(TARGETS))]
    
    with torch.no_grad():
        for batch in loader:
            out = torch.sigmoid(model(batch))
            y = batch.y.view(-1, len(TARGETS))
            for i in range(len(TARGETS)):
                for j in range(y.shape[0]):
                    label = y[j, i].item()
                    if label != -1:
                        all_preds[i].append(out[j, i].item())
                        all_labels[i].append(label)
    
    aucs = []
    for i, t in enumerate(TARGETS):
        if len(set(all_labels[i])) > 1:
            auc = roc_auc_score(all_labels[i], all_preds[i])
            aucs.append(auc)
        else:
            aucs.append(None)
    return aucs


optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training with early stopping
best_mean_auc = 0
patience = 10
no_improve = 0
best_state = None

print("Training multi-task GNN...\n")
for epoch in range(1, 101):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(batch)
        loss = masked_loss(out, batch.y, pos_weight_tensor)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if epoch % 5 == 0:
        aucs = evaluate_multitask(test_loader)
        valid_aucs = [a for a in aucs if a is not None]
        mean_auc = np.mean(valid_aucs)
        
        if mean_auc > best_mean_auc:
            best_mean_auc = mean_auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
            print(f"Epoch {epoch:3d} | Loss: {total_loss/len(train_loader):.4f} | Mean AUC: {mean_auc:.4f} ✓")
        else:
            no_improve += 1
            print(f"Epoch {epoch:3d} | Loss: {total_loss/len(train_loader):.4f} | Mean AUC: {mean_auc:.4f}")
            if no_improve >= patience:
                print(f"\nEarly stopping at epoch {epoch}")
                break

print(f"\nBest Mean AUC across all 12 targets: {best_mean_auc:.4f}")

Training multi-task GNN...

Epoch   5 | Loss: 1.1401 | Mean AUC: 0.6908 ✓
Epoch  10 | Loss: 1.1226 | Mean AUC: 0.7004 ✓
Epoch  15 | Loss: 1.1067 | Mean AUC: 0.7117 ✓
Epoch  20 | Loss: 1.0927 | Mean AUC: 0.7223 ✓
Epoch  25 | Loss: 1.0915 | Mean AUC: 0.7252 ✓
Epoch  30 | Loss: 1.0796 | Mean AUC: 0.7354 ✓
Epoch  35 | Loss: 1.0618 | Mean AUC: 0.7385 ✓
Epoch  40 | Loss: 1.0555 | Mean AUC: 0.7394 ✓
Epoch  45 | Loss: 1.0484 | Mean AUC: 0.7403 ✓
Epoch  50 | Loss: 1.0387 | Mean AUC: 0.7363
Epoch  55 | Loss: 1.0383 | Mean AUC: 0.7449 ✓
Epoch  60 | Loss: 1.0291 | Mean AUC: 0.7456 ✓
Epoch  65 | Loss: 1.0279 | Mean AUC: 0.7499 ✓
Epoch  70 | Loss: 1.0129 | Mean AUC: 0.7471
Epoch  75 | Loss: 1.0153 | Mean AUC: 0.7498
Epoch  80 | Loss: 1.0010 | Mean AUC: 0.7553 ✓
Epoch  85 | Loss: 0.9993 | Mean AUC: 0.7650 ✓
Epoch  90 | Loss: 0.9966 | Mean AUC: 0.7627
Epoch  95 | Loss: 0.9868 | Mean AUC: 0.7659 ✓
Epoch 100 | Loss: 0.9775 | Mean AUC: 0.7666 ✓

Best Mean AUC across all 12 targets: 0.7666


In [10]:
print("Continuing training...\n")
for epoch in range(101, 201):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(batch)
        loss = masked_loss(out, batch.y, pos_weight_tensor)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if epoch % 5 == 0:
        aucs = evaluate_multitask(test_loader)
        valid_aucs = [a for a in aucs if a is not None]
        mean_auc = np.mean(valid_aucs)
        
        if mean_auc > best_mean_auc:
            best_mean_auc = mean_auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
            print(f"Epoch {epoch:3d} | Loss: {total_loss/len(train_loader):.4f} | Mean AUC: {mean_auc:.4f} ✓")
        else:
            no_improve += 1
            print(f"Epoch {epoch:3d} | Loss: {total_loss/len(train_loader):.4f} | Mean AUC: {mean_auc:.4f}")
            if no_improve >= patience:
                print(f"\nEarly stopping at epoch {epoch}")
                break

print(f"\nBest Mean AUC across all 12 targets: {best_mean_auc:.4f}")

Continuing training...

Epoch 105 | Loss: 0.9739 | Mean AUC: 0.7660
Epoch 110 | Loss: 0.9682 | Mean AUC: 0.7740 ✓
Epoch 115 | Loss: 0.9580 | Mean AUC: 0.7752 ✓
Epoch 120 | Loss: 0.9543 | Mean AUC: 0.7792 ✓
Epoch 125 | Loss: 0.9447 | Mean AUC: 0.7767
Epoch 130 | Loss: 0.9495 | Mean AUC: 0.7844 ✓
Epoch 135 | Loss: 0.9451 | Mean AUC: 0.7827
Epoch 140 | Loss: 0.9391 | Mean AUC: 0.7853 ✓
Epoch 145 | Loss: 0.9257 | Mean AUC: 0.7889 ✓
Epoch 150 | Loss: 0.9267 | Mean AUC: 0.7840
Epoch 155 | Loss: 0.9122 | Mean AUC: 0.7890 ✓
Epoch 160 | Loss: 0.9140 | Mean AUC: 0.7869
Epoch 165 | Loss: 0.9149 | Mean AUC: 0.7923 ✓
Epoch 170 | Loss: 0.9057 | Mean AUC: 0.7869
Epoch 175 | Loss: 0.9103 | Mean AUC: 0.7868
Epoch 180 | Loss: 0.9001 | Mean AUC: 0.7893
Epoch 185 | Loss: 0.8906 | Mean AUC: 0.7816
Epoch 190 | Loss: 0.8930 | Mean AUC: 0.7926 ✓
Epoch 195 | Loss: 0.8907 | Mean AUC: 0.7854
Epoch 200 | Loss: 0.8806 | Mean AUC: 0.7857

Best Mean AUC across all 12 targets: 0.7926


In [11]:
print("Continuing training...\n")
no_improve = 0  # reset patience counter

for epoch in range(201, 351):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(batch)
        loss = masked_loss(out, batch.y, pos_weight_tensor)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if epoch % 5 == 0:
        aucs = evaluate_multitask(test_loader)
        valid_aucs = [a for a in aucs if a is not None]
        mean_auc = np.mean(valid_aucs)
        
        if mean_auc > best_mean_auc:
            best_mean_auc = mean_auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
            print(f"Epoch {epoch:3d} | Loss: {total_loss/len(train_loader):.4f} | Mean AUC: {mean_auc:.4f} ✓")
        else:
            no_improve += 1
            print(f"Epoch {epoch:3d} | Loss: {total_loss/len(train_loader):.4f} | Mean AUC: {mean_auc:.4f}")
            if no_improve >= patience:
                print(f"\nEarly stopping at epoch {epoch}")
                break

print(f"\nBest Mean AUC across all 12 targets: {best_mean_auc:.4f}")

Continuing training...

Epoch 205 | Loss: 0.8800 | Mean AUC: 0.7847
Epoch 210 | Loss: 0.8731 | Mean AUC: 0.7913
Epoch 215 | Loss: 0.8731 | Mean AUC: 0.7887
Epoch 220 | Loss: 0.8726 | Mean AUC: 0.7913
Epoch 225 | Loss: 0.8644 | Mean AUC: 0.7884
Epoch 230 | Loss: 0.8645 | Mean AUC: 0.7890
Epoch 235 | Loss: 0.8589 | Mean AUC: 0.7917
Epoch 240 | Loss: 0.8612 | Mean AUC: 0.7890
Epoch 245 | Loss: 0.8515 | Mean AUC: 0.7917
Epoch 250 | Loss: 0.8609 | Mean AUC: 0.7932 ✓
Epoch 255 | Loss: 0.8581 | Mean AUC: 0.7879
Epoch 260 | Loss: 0.8472 | Mean AUC: 0.7913
Epoch 265 | Loss: 0.8665 | Mean AUC: 0.7964 ✓
Epoch 270 | Loss: 0.8415 | Mean AUC: 0.7952
Epoch 275 | Loss: 0.8335 | Mean AUC: 0.7911
Epoch 280 | Loss: 0.8378 | Mean AUC: 0.7913
Epoch 285 | Loss: 0.8345 | Mean AUC: 0.7939
Epoch 290 | Loss: 0.8265 | Mean AUC: 0.7941
Epoch 295 | Loss: 0.8342 | Mean AUC: 0.7967 ✓
Epoch 300 | Loss: 0.8136 | Mean AUC: 0.7914
Epoch 305 | Loss: 0.8248 | Mean AUC: 0.7963
Epoch 310 | Loss: 0.8042 | Mean AUC: 0.7964
Ep

In [12]:
# Load best weights
model.load_state_dict(best_state)

# Per-target AUC breakdown
aucs = evaluate_multitask(test_loader)
print("Per-target AUC-ROC on test set:\n")
print(f"{'Target':<20} {'AUC':>6}")
print("-" * 28)
for t, auc in zip(TARGETS, aucs):
    if auc is not None:
        print(f"{t:<20} {auc:.4f}")
    else:
        print(f"{t:<20}    N/A")

valid_aucs = [a for a in aucs if a is not None]
print("-" * 28)
print(f"{'Mean AUC':<20} {sum(valid_aucs)/len(valid_aucs):.4f}")

# Save model
torch.save(best_state, "best_multitask_gnn.pt")
print("\nModel saved!")

Per-target AUC-ROC on test set:

Target                  AUC
----------------------------
NR-AR                0.7887
NR-AR-LBD            0.8379
NR-AhR               0.8344
NR-Aromatase         0.7862
NR-ER                0.6608
NR-ER-LBD            0.7596
NR-PPAR-gamma        0.7906
SR-ARE               0.7845
SR-ATAD5             0.8483
SR-HSE               0.7592
SR-MMP               0.8776
SR-p53               0.8328
----------------------------
Mean AUC             0.7967

Model saved!


In [13]:
import json

results = {t: round(auc, 4) if auc else None 
           for t, auc in zip(TARGETS, aucs)}
results["mean_auc"] = round(sum(valid_aucs)/len(valid_aucs), 4)

with open("results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Results saved!")

Results saved!
